In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

In [ ]:
# hyper

latent_dim = 100
batch_size = 128
lr = 0.0002
epochs = 50
device = torch.device("cuda")

In [ ]:
device

In [ ]:
# dataloader to transform the incoming tensor
transform = transforms.Compose([
     transforms.ToTensor(),
     transforms.Normalize([0.5],[0.5])
])

In [ ]:
dataloader = torch.utils.data.DataLoader(
    torchvision.datasets.MNIST("./data",train=True,download=True,transform=transform),
    batch_size=batch_size,
    shuffle=True
)

In [ ]:
# Generator Network   R^100 -> R^784
class Generator(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(latent_dim,128),
        nn.ReLU(True),
        nn.Linear(128,256),
        # Adding BatchNorm for better training stability,
        # 局部归一化，经验上越早用BN越早稳住梯度
        # 不要放在输入层后，也不要放在输出层Tanh()前
        # 只在“关键中间层”用 BN，而不是层层都用
        nn.BatchNorm1d(256),
        nn.ReLU(True),
        nn.Linear(256,512),
        nn.ReLU(True),
        nn.Linear(512,784),
        nn.Tanh()
    )

  def forward(self,x):
    return self.net(x).view(-1,1,28,28)


# Discriminator Network  R^784 -> R^1
class Discriminator(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(784,512),
        nn.LeakyReLU(0.2),
        nn.Linear(512,256),
        nn.LeakyReLU(0.2),
        nn.Linear(256,1),
        nn.Sigmoid()
    )

  def forward(self,img):
    return self.net(img.view(img.size(0),-1))


In [ ]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(),lr=lr,betas=(0.5,0.999))
optimizer_D = optim.Adam(discriminator.parameters(),lr=lr,betas=(0.5,0.999))


In [ ]:
# Training loop
for epoch in range(epochs):
  # len(dataloader) = 60000/128 ≈ 469
  for i,(real_imgs,_) in enumerate(dataloader):
    # real_imgs: [128,1,28,28]
    real_imgs = real_imgs.to(device)
    batch_size = real_imgs.size(0)

    # 128个一维标签(1)
    real_labels = torch.ones(batch_size,1).to(device)
    # 128个一维标签(0)
    fake_labels = torch.zeros(batch_size,1).to(device)

    # Train D   “最大化识别真图片为1、假图片为0的能力”
    z = torch.randn(batch_size,latent_dim).to(device)
    fake_imgs = generator(z) #这一步会自动调用nn.Module.forward()

    # 计算D的损失函数
    real_loss = criterion(discriminator(real_imgs),real_labels)
    fake_loss = criterion(discriminator(fake_imgs.detach()),fake_labels)
    d_loss = real_loss + fake_loss

    '''优化器D的梯度清零、反向传播和参数更新'''
    optimizer_D.zero_grad()   #清空上一次的梯度
    d_loss.backward()         #计算梯度
    optimizer_D.step()        #更新参数：根据梯度调整权重

    # Train G   “最大化让D认为假图片为1的能力”
    z = torch.randn(batch_size,latent_dim).to(device)
    # 用当前的G生成假图片
    generated_imgs = generator(z)
    # 计算G的损失函数
    g_loss = criterion(discriminator(generated_imgs),real_labels)

    '''优化器G的梯度清零、反向传播和参数更新'''
    optimizer_G.zero_grad()
    g_loss.backward()
    optimizer_G.step()

    if i%200==0:
      print(f"Epoch {epoch}/{epochs} Batch {i}/{len(dataloader)} LossD:{d_loss.item():.4f}, LossG:{g_loss.item():.4f}")

  # 每个epoch结束后，用当前程度的G生成一些图片进行展示
  with torch.no_grad():
    fake = generator(torch.randn(64,latent_dim).to(device)).detach().cpu()
    grid = make_grid(fake,nrow=8,normalize=True)
    plt.imshow(grid.permute(1,2,0).numpy())
    plt.title(f"Epoch {epoch}")
    plt.axis("off")
    plt.show()

